In [ ]:
!pip install transformers torch sentencepiece accelerate bitsandbytes scikit-learn pandas plotly umap-learn

In [1]:
# %%time
# Магическая команда %%time (опционально) покажет общее время выполнения ячейки.

# ==============================================================================
# 1. ИМПОРТ БИБЛИОТЕК И НАСТРОЙКА
# ==============================================================================
import torch
from transformers import AutoModel, AutoTokenizer
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
from tqdm.notebook import tqdm
import sys

# Проверяем наличие cuML и настраиваем его.
try:
    import cuml
    import cupy as cp

    print(
        f"Библиотека cuML версии {cuml.__version__} найдена. Вычисления будут на GPU."
    )
except ImportError:
    print("КРИТИЧЕСКАЯ ОШИБКА: Библиотека cuML не найдена.")
    raise

# Настраиваем рендеринг для plotly в среде Jupyter.
pio.renderers.default = "jupyterlab"
MODEL_NAME = "Qwen/Qwen2-1.5B-Instruct"

# ==============================================================================
#   <<<<<  ПАРАМЕТРЫ ВИЗУАЛИЗАЦИИ (РЕДАКТИРОВАТЬ ЗДЕСЬ)  >>>>>
# ==============================================================================
PERCENT_TO_VIEW = 95.0
NUM_HIGHLIGHT_POINTS = 10  # Количество точек для подсветки
RANDOM_SEED = 42  # Для воспроизводимости случайного выбора


# ==============================================================================
# 2. ЗАГРУЗКА МОДЕЛИ И ТОКЕНИЗАТОРА (без изменений)
# ==============================================================================
print(f"\nЗагрузка модели и токенизатора '{MODEL_NAME}'...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_NAME, trust_remote_code=True, torch_dtype="auto", device_map="auto"
)
print("Модель и токенизатор успешно загружены.")


# ==============================================================================
# 3. ИЗВЛЕЧЕНИЕ И ПОДГОТОВКА ДАННЫХ (без изменений)
# ==============================================================================
print("\nИзвлечение и подготовка матрицы эмбеддингов...")
full_embedding_matrix = model.get_input_embeddings().weight.detach()
vocab = tokenizer.get_vocab()
vocab_size = len(vocab)
embeddings_for_viz = full_embedding_matrix[:vocab_size, :]
embeddings_tensor_float32 = embeddings_for_viz.to(torch.float32)
all_tokens = [""] * vocab_size
for token, token_id in tqdm(vocab.items(), desc="Сопоставление токенов"):
    if token_id < vocab_size:
        all_tokens[token_id] = token
print("Данные готовы.")


# ==============================================================================
# 4. СНИЖЕНИЕ РАЗМЕРНОСТИ (UMAP НА GPU) (без изменений)
# ==============================================================================
print(f"\nСнижение размерности ВСЕХ {vocab_size} токенов с помощью UMAP на GPU...")
embeddings_cupy = cp.asarray(embeddings_tensor_float32)
umap_3d = cuml.UMAP(
    n_components=3, n_neighbors=15, min_dist=0.1, metric="cosine", verbose=True
)
embeddings_3d_gpu = umap_3d.fit_transform(embeddings_cupy)
embeddings_3d_cpu = embeddings_3d_gpu.get()
print("Снижение размерности успешно завершено.")


# ==============================================================================
# 5. АВТОМАТИЧЕСКИЙ РАСЧЕТ ОПТИМАЛЬНОГО СРЕЗА (без изменений)
# ==============================================================================
print(f"\nАвтоматический расчет среза для отображения {PERCENT_TO_VIEW}% точек...")
lower_percentile = (100 - PERCENT_TO_VIEW) / 2
upper_percentile = 100 - lower_percentile
x_min, x_max = np.percentile(
    embeddings_3d_cpu[:, 0], [lower_percentile, upper_percentile]
)
y_min, y_max = np.percentile(
    embeddings_3d_cpu[:, 1], [lower_percentile, upper_percentile]
)
z_min, z_max = np.percentile(
    embeddings_3d_cpu[:, 2], [lower_percentile, upper_percentile]
)
max_len = max(x_max - x_min, y_max - y_min, z_max - z_min)
x_center, y_center, z_center = np.median(embeddings_3d_cpu, axis=0)
half_len = max_len / 2
X_RANGE = [x_center - half_len, x_center + half_len]
Y_RANGE = [y_center - half_len, y_center + half_len]
Z_RANGE = [z_center - half_len, z_center + half_len]
print("Расчет завершен.")


# ==============================================================================
# 6. ВИЗУАЛИЗАЦИЯ С ПОДСВЕТКОЙ ТОЧЕК
# ==============================================================================
print(f"\nСоздание визуализации с подсветкой {NUM_HIGHLIGHT_POINTS} случайных точек...")

# --- НОВЫЙ БЛОК: ПОДГОТОВКА ДАННЫХ ДЛЯ ПОДСВЕТКИ ---
np.random.seed(RANDOM_SEED)
highlight_indices = np.random.choice(vocab_size, NUM_HIGHLIGHT_POINTS, replace=False)

# Данные для выделенных точек
highlight_coords = embeddings_3d_cpu[highlight_indices]
highlight_tokens = [all_tokens[i] for i in highlight_indices]
# ----------------------------------------------------

# Создаем два слоя (трейса) для графика
trace_background = go.Scatter3d(
    x=embeddings_3d_cpu[:, 0],
    y=embeddings_3d_cpu[:, 1],
    z=embeddings_3d_cpu[:, 2],
    mode="markers",
    hoverinfo="none",  # Отключаем подсказки для фона
    marker=dict(
        size=1.5,
        color="lightgray",  # Нейтральный серый цвет
        opacity=0.3,  # Делаем фон полупрозрачным
    ),
    name="Все токены",  # Имя для легенды
)

trace_highlight = go.Scatter3d(
    x=highlight_coords[:, 0],
    y=highlight_coords[:, 1],
    z=highlight_coords[:, 2],
    mode="markers",
    text=highlight_tokens,
    hovertemplate=(
        "<b>Токен:</b> %{text}<br><br>"
        + "<b>Координаты:</b><br>"
        + "X: %{x:.3f}<br>Y: %{y:.3f}<br>Z: %{z:.3f}"
        + "<extra></extra>"
    ),
    marker=dict(
        size=5,  # Делаем точки крупнее
        color="lime",  # Яркий зеленый цвет
        opacity=1.0,
        line=dict(width=1, color="black"),  # Добавляем обводку для лучшей видимости
    ),
    name="Выделенные",  # Имя для легенды
)

# Создаем фигуру, передавая оба слоя в виде списка
fig = go.Figure(data=[trace_background, trace_highlight])

fig.update_layout(
    title=f"Оптимальный срез ({PERCENT_TO_VIEW}%) с подсветкой {NUM_HIGHLIGHT_POINTS} точек",
    margin=dict(l=0, r=0, b=0, t=40),
    height=800,
    scene=dict(
        xaxis=dict(range=X_RANGE, title="Компонента 1"),
        yaxis=dict(range=Y_RANGE, title="Компонента 2"),
        zaxis=dict(range=Z_RANGE, title="Компонента 3"),
        aspectmode="cube",
        bgcolor="rgb(20, 24, 54)",
    ),
    hoverlabel=dict(bgcolor="white", font_size=14),
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01),  # Показываем легенду
)

fig.show()

Библиотека cuML версии 25.06.00 найдена. Вычисления будут на GPU.

Загрузка модели и токенизатора 'Qwen/Qwen2-1.5B-Instruct'...
Модель и токенизатор успешно загружены.

Извлечение и подготовка матрицы эмбеддингов...


Сопоставление токенов:   0%|          | 0/151646 [00:00<?, ?it/s]

Данные готовы.

Снижение размерности ВСЕХ 151646 токенов с помощью UMAP на GPU...
[2025-08-06 19:59:25.831] [CUML] [debug] Computing KNN Graph
[2025-08-06 19:59:26.869] [CUML] [debug] Computing fuzzy simplicial set
Снижение размерности успешно завершено.

Автоматический расчет среза для отображения 95.0% точек...
Расчет завершен.

Создание визуализации с подсветкой 10 случайных точек...
